# A. Biodata Mahasiswa
## Mata Kuliah: Dasar Ilmu Data (GIK2GAB3)
## Materi: Tugas Besar - Model Support Vector Machines (SVM) & Hyperparameter Tuning pada Studi Kasus Klasifikasi menggunakan Dataset Bank Marketing

*   NIM   : 707012400017
*   Nama  : Falih Afnan Fauzi
*   Kelas : 48-03

**Sumber dataset:**
- Bank Marketing Dataset (UCI Machine Learning Repository)
- Sumber: S. Moro, P. Cortez and P. Rita. A Data-Driven Approach to Predict the Success of Bank Telemarketing. Decision Support Systems, 2014.

**Tujuan:**
- Melakukan prediksi pada studi kasus klasifikasi dengan menggunakan model SVM (Support Vector Machine)
- Prediksi klasifikasi yang dilakukan adalah memprediksi keputusan nasabah: Apakah nasabah akan melakukan **deposit** (`yes`) atau **tidak** (`no`)


# B. Load Library dan Dataset


## B1. Import Library


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
import time
import warnings
warnings.filterwarnings('ignore')

# Set style untuk visualisasi
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## B2. Load Dataset


In [ ]:
# Load dataset Bank Marketing
df = pd.read_csv('archive/bank.csv')

# Menampilkan 10 data teratas
df.head(10)

**Informasi Dataset**

Dataset **Bank Marketing** berasal dari kampanye pemasaran langsung (telepon) yang dilakukan oleh sebuah institusi perbankan di Portugal. Tujuannya adalah untuk memprediksi apakah nasabah akan berlangganan deposito berjangka (`deposit`).

- Dataset memiliki **16 fitur** dan **1 label diskret (`deposit`: yes/no)**.
- Fitur terdiri dari informasi demografis nasabah dan data kampanye pemasaran:

| No | Fitur | Tipe | Keterangan |
|---|---|---|---|
| 1 | `age` | Numerik | Umur nasabah |
| 2 | `job` | Kategorikal | Jenis pekerjaan |
| 3 | `marital` | Kategorikal | Status pernikahan |
| 4 | `education` | Kategorikal | Tingkat pendidikan |
| 5 | `default` | Kategorikal | Memiliki kredit macet? (yes/no) |
| 6 | `balance` | Numerik | Saldo rata-rata tahunan (euro) |
| 7 | `housing` | Kategorikal | Memiliki kredit rumah? (yes/no) |
| 8 | `loan` | Kategorikal | Memiliki pinjaman personal? (yes/no) |
| 9 | `contact` | Kategorikal | Tipe komunikasi kontak |
| 10 | `day` | Numerik | Hari terakhir dihubungi dalam sebulan |
| 11 | `month` | Kategorikal | Bulan terakhir dihubungi |
| 12 | `duration` | Numerik | Durasi kontak terakhir (detik) |
| 13 | `campaign` | Numerik | Jumlah kontak selama kampanye ini |
| 14 | `pdays` | Numerik | Jumlah hari sejak kontak terakhir dari kampanye sebelumnya |
| 15 | `previous` | Numerik | Jumlah kontak sebelum kampanye ini |
| 16 | `poutcome` | Kategorikal | Hasil kampanye pemasaran sebelumnya |


In [ ]:
# Menampilkan informasi dataset
df.info()

In [ ]:
# Menampilkan statistik deskriptif
df.describe()

In [ ]:
# Melihat distribusi label (deposit)
print('Distribusi label (deposit):')
print(df['deposit'].value_counts())
print(f'\nPersentase:')
print(df['deposit'].value_counts(normalize=True) * 100)

In [ ]:
# Visualisasi distribusi label
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#2ecc71']
df['deposit'].value_counts().plot(kind='bar', color=colors, edgecolor='black', ax=ax)
ax.set_title('Distribusi Label Deposit', fontsize=14, fontweight='bold')
ax.set_xlabel('Deposit', fontsize=12)
ax.set_ylabel('Jumlah', fontsize=12)
ax.set_xticklabels(['No', 'Yes'], rotation=0)
for i, v in enumerate(df['deposit'].value_counts().values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('plots/distribusi_label.png', dpi=150, bbox_inches='tight')
plt.show()

## B3. Preprocessing Data

Dataset ini memiliki kolom kategorikal yang perlu di-encode menjadi numerik agar dapat diproses oleh model SVM. Selain itu, SVM sangat sensitif terhadap skala fitur, sehingga normalisasi/standarisasi data diperlukan.


In [ ]:
# Melihat kolom kategorikal
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Kolom kategorikal:', categorical_cols)
print(f'Jumlah kolom kategorikal: {len(categorical_cols)}')

# Melihat nilai unik pada setiap kolom kategorikal
for col in categorical_cols:
    print(f'\n{col}: {df[col].unique()}')

In [ ]:
# Cek missing values
print('Missing values per kolom:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Capping Outliers menggunakan metode IQR
outlier_cols = ['balance', 'duration', 'campaign', 'pdays', 'previous']
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Outliers berhasil di-cap menggunakan nilai batas IQR.")


In [ ]:
# Label Encoding untuk kolom kategorikal
df_encoded = df.copy()
le_dict = {}  # Simpan encoder untuk referensi

categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

for col in categorical_features:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    le_dict[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Encode label target
le_target = LabelEncoder()
df_encoded['deposit'] = le_target.fit_transform(df_encoded['deposit'])
print(f'\ndeposit: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

print('\n--- Dataset setelah encoding ---')
df_encoded.head()

# C. Klasifikasi dengan SVM


## C1. Tahap 1: Tentukan Fitur dan Label


In [ ]:
# Menentukan fitur (X) dan label (y)
# Fitur = semua kolom kecuali 'deposit'
X = df_encoded.drop('deposit', axis=1)
# Label = kolom 'deposit'
y = df_encoded['deposit']

print(f'Jumlah fitur: {X.shape[1]}')
print(f'Nama fitur: {list(X.columns)}')
print(f'Jumlah data: {X.shape[0]}')

**Q1. Jelaskan penentuan fitur dan label. Mengapa anda memilih kolom `deposit` sebagai label? Mengapa anda memilih kolom lainnya sebagai fitur?**

**Jawaban:**

- **Label** yang dipilih adalah kolom `deposit` karena kolom ini merupakan variabel target yang ingin diprediksi, yaitu apakah nasabah akan melakukan deposit berjangka (`yes` = 1) atau tidak (`no` = 0). Kolom `deposit` bersifat **diskret/kategorikal** sehingga cocok untuk studi kasus **klasifikasi**.

- **Fitur** yang digunakan adalah 16 kolom lainnya (`age`, `job`, `marital`, `education`, `default`, `balance`, `housing`, `loan`, `contact`, `day`, `month`, `duration`, `campaign`, `pdays`, `previous`, `poutcome`) karena kolom-kolom tersebut berisi informasi demografis nasabah dan data riwayat kampanye pemasaran yang menjadi **indikator/variabel prediktor** untuk memprediksi keputusan deposit nasabah.


In [ ]:
# Menampilkan jumlah data pada setiap kelas
print('Jumlah data per kelas:')
print(y.value_counts())
print(f'\nKelas 0 (No): {(y == 0).sum()} data')
print(f'Kelas 1 (Yes): {(y == 1).sum()} data')

In [ ]:
# Menampilkan data label
print('Data Label (y):')
print(y)
print(f'\nShape: {y.shape}')

In [ ]:
# Menampilkan fitur
print('Data Fitur (X):')
X

**Q2. Jelaskan hasil tampilan label dan fitur di atas**

**Jawaban:**

- **Label (y):** Berisi 11.162 data dengan 2 kelas, yaitu `0` (No/tidak deposit) dan `1` (Yes/deposit). Data label sudah di-encode dari nilai string (`yes`/`no`) menjadi numerik (`1`/`0`) menggunakan `LabelEncoder`.

- **Fitur (X):** Terdiri dari 11.162 baris dan 16 kolom. Semua kolom kategorikal sudah di-encode menjadi numerik menggunakan `LabelEncoder`. Fitur numerik asli (`age`, `balance`, `day`, `duration`, `campaign`, `pdays`, `previous`) tetap dalam bentuk aslinya. Fitur kategorikal (`job`, `marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, `poutcome`) sudah diubah menjadi angka.


## C2. Tahap 2: Bagi Dataset menjadi Data Latih (Train Data) dan Data Uji (Test Data)


In [ ]:
# Membagi dataset menjadi data latih (80%) dan data uji (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Jumlah total data: {len(X)}')
print(f'Jumlah data latih: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Jumlah data uji: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)')

In [ ]:
# Standarisasi fitur menggunakan StandardScaler
# SVM sangat sensitif terhadap skala fitur, sehingga standarisasi diperlukan
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Standarisasi fitur berhasil dilakukan.')
print(f'\nMean fitur sebelum scaling (train): {X_train.mean().values[:5]}...')
print(f'Mean fitur setelah scaling (train): {X_train_scaled.mean(axis=0)[:5].round(4)}...')
print(f'\nStd fitur sebelum scaling (train): {X_train.std().values[:5]}...')
print(f'Std fitur setelah scaling (train): {X_train_scaled.std(axis=0)[:5].round(4)}...')

In [ ]:
# Menampilkan data latih dan data uji
print('=== Data Latih ===')
print(f'X_train shape: {X_train_scaled.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Distribusi kelas y_train:')
print(y_train.value_counts())

print('\n=== Data Uji ===')
print(f'X_test shape: {X_test_scaled.shape}')
print(f'y_test shape: {y_test.shape}')
print(f'Distribusi kelas y_test:')
print(y_test.value_counts())

**Q3. Jelaskan hasil penentuan jumlah data latih dan data uji**

**Jawaban:**

Dataset dibagi menggunakan rasio **80:20**, yaitu:
- **Data latih (train):** 8.929 data (80%) — digunakan untuk melatih model SVM agar mempelajari pola dari data.
- **Data uji (test):** 2.233 data (20%) — digunakan untuk menguji performa model pada data yang belum pernah dilihat sebelumnya.

Parameter `stratify=y` digunakan agar proporsi kelas pada data latih dan data uji tetap seimbang mengikuti proporsi kelas pada dataset asli. Hal ini penting agar model tidak bias terhadap salah satu kelas.

Selain itu, dilakukan **standarisasi (StandardScaler)** pada fitur karena SVM sangat sensitif terhadap skala fitur. StandardScaler mengubah setiap fitur agar memiliki mean = 0 dan standard deviation = 1. Penting: `fit_transform()` hanya dilakukan pada data latih, sedangkan `transform()` pada data uji, untuk menghindari **data leakage**.


## C3. Tahap 3: Siapkan Classifier dan Tentukan Variabel/Parameternya


In [ ]:
# Menentukan variabel classifier SVM (tanpa HPO)
# Menggunakan parameter default dengan kernel RBF
classifier = SVC(
    kernel='rbf',       # Kernel Radial Basis Function (default)
    C=1.0,              # Parameter regularisasi (default)
    gamma='scale',      # Koefisien kernel (default)
    random_state=42     # Untuk reproduktibilitas hasil
)

print('Classifier SVM (tanpa HPO):')
print(classifier)

**Q4. Jelaskan hasil dan penentuan variabel classifier yang anda buat!**

**Jawaban:**

Classifier SVM dibuat menggunakan `SVC()` (Support Vector Classifier) dari library scikit-learn dengan parameter sebagai berikut:

- **`kernel='rbf'`**: Menggunakan kernel Radial Basis Function (RBF), yang merupakan kernel default dan paling umum digunakan. Kernel RBF mampu menangani data yang tidak dapat dipisahkan secara linier dengan memetakan data ke ruang dimensi yang lebih tinggi.

- **`C=1.0`**: Parameter regularisasi (penalty) dengan nilai default 1.0. Parameter C mengontrol trade-off antara memaksimalkan margin dan meminimalkan kesalahan klasifikasi. Nilai C yang lebih besar membuat model lebih ketat (less tolerant terhadap misklasifikasi).

- **`gamma='scale'`**: Koefisien kernel RBF dengan nilai default `'scale'`, yang berarti gamma = 1 / (n_features × variance_fitur). Gamma menentukan seberapa jauh pengaruh satu sampel data.

- **`random_state=42`**: Digunakan untuk memastikan hasil yang dapat direproduksi (reproducible).


## C4. Tahap 4: Lakukan Proses Training dengan Data Latih


In [ ]:
# Melakukan training model SVM dengan data latih
start_time = time.time()
classifier.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

print(f'Training selesai!')
print(f'Waktu training: {training_time:.2f} detik')
print(f'Jumlah support vectors: {classifier.n_support_}')
print(f'Total support vectors: {sum(classifier.n_support_)}')

**Q5. Jelaskan code training yang anda buat!**

**Jawaban:**

Proses training dilakukan dengan memanggil method `classifier.fit(X_train_scaled, y_train)`, dimana:
- `X_train_scaled`: Data fitur latih yang sudah distandarisasi, berisi 8.929 data dengan 16 fitur.
- `y_train`: Data label latih yang berisi kelas target (0 = No, 1 = Yes).

Pada proses training, model SVM mempelajari pola dari data latih untuk menemukan **hyperplane** terbaik yang memisahkan kedua kelas. SVM juga mengidentifikasi **support vectors**, yaitu titik-titik data yang paling dekat dengan hyperplane dan menjadi penentu posisi hyperplane optimal.

Waktu training juga dicatat untuk keperluan perbandingan performa nantinya.


## C5. Tahap 5: Lakukan Pengujian dengan Data Uji


In [ ]:
# Melakukan prediksi pada data uji
y_pred = classifier.predict(X_test_scaled)

# Perbandingan data prediksi dengan label asli
comparison = pd.DataFrame({
    'Data Ke': range(1, len(y_test) + 1),
    'Label Asli': y_test.values,
    'Hasil Prediksi': y_pred,
    'Keterangan': ['Benar' if a == p else 'Salah' for a, p in zip(y_test.values, y_pred)]
})

print('Perbandingan Label Asli vs Hasil Prediksi (20 data pertama):')
comparison.head(20)

In [ ]:
# Menghitung jumlah prediksi benar dan salah
benar = (comparison['Keterangan'] == 'Benar').sum()
salah = (comparison['Keterangan'] == 'Salah').sum()

print(f'Jumlah prediksi benar: {benar} dari {len(y_test)} data')
print(f'Jumlah prediksi salah: {salah} dari {len(y_test)} data')

**Q6. Jelaskan hasil pengujian yang anda dapatkan!**

**Jawaban:**

Pengujian dilakukan dengan memanggil `classifier.predict(X_test_scaled)` untuk memprediksi label pada 2.233 data uji. Hasilnya dibandingkan dengan label asli (`y_test`) dalam bentuk tabel perbandingan.

Dari tabel perbandingan, dapat dilihat bahwa sebagian besar prediksi model SVM **sesuai** dengan label asli (ditandai 'Benar'), meskipun ada beberapa data yang diprediksi salah (ditandai 'Salah'). Ini menunjukkan bahwa model SVM tanpa HPO sudah cukup baik dalam memprediksi keputusan deposit nasabah, namun masih ada ruang untuk peningkatan melalui optimasi hyperparameter.


## C6. Tahap 6: Analisa Performansi Model


### C6.1 Menggunakan Accuracy Score


In [ ]:
# Menghitung skor akurasi
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy Score (tanpa HPO): {accuracy:.4f}')
print(f'Accuracy Percentage: {accuracy * 100:.2f}%')

**Q7. Jelaskan hasil performansi menggunakan accuracy score yang anda dapatkan!**

**Jawaban:**

Accuracy score mengukur proporsi prediksi yang benar dari total keseluruhan prediksi. Dengan rumus:

$$Accuracy = \frac{\text{Jumlah Prediksi Benar}}{\text{Jumlah Total Data Uji}}$$

Hasil accuracy score yang diperoleh dari model SVM tanpa HPO menunjukkan seberapa besar persentase data uji yang berhasil diklasifikasikan dengan benar. Semakin tinggi nilai accuracy (mendekati 1.0 atau 100%), semakin baik performa model. Nilai accuracy ini akan menjadi baseline untuk dibandingkan dengan model yang menggunakan HPO.


### C6.2 Menggunakan Confusion Matrix


In [ ]:
# Menampilkan confusion matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)

# Visualisasi confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No (0)', 'Yes (1)'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Confusion Matrix - SVM tanpa HPO', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrix_tanpa_hpo.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretasi confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negative (TN): {tn} — Prediksi No, Aktual No (Benar)')
print(f'False Positive (FP): {fp} — Prediksi Yes, Aktual No (Salah)')
print(f'False Negative (FN): {fn} — Prediksi No, Aktual Yes (Salah)')
print(f'True Positive (TP): {tp} — Prediksi Yes, Aktual Yes (Benar)')

**Q8. Jelaskan hasil performansi menggunakan confusion matrix yang anda dapatkan!**

**Jawaban:**

Confusion matrix adalah tabel yang menunjukkan distribusi hasil prediksi model dibandingkan dengan label sebenarnya. Tabel terdiri dari 4 komponen:

- **True Negative (TN):** Jumlah data yang benar diprediksi sebagai kelas negatif (No deposit) — model benar.
- **False Positive (FP):** Jumlah data yang salah diprediksi sebagai kelas positif (Yes deposit) padahal sebenarnya negatif — model salah ("salah tuduh").
- **False Negative (FN):** Jumlah data yang salah diprediksi sebagai kelas negatif (No deposit) padahal sebenarnya positif — model salah ("yang lolos").
- **True Positive (TP):** Jumlah data yang benar diprediksi sebagai kelas positif (Yes deposit) — model benar.

Dari confusion matrix di atas, dapat dilihat bahwa model SVM tanpa HPO memiliki performa yang baik ditunjukkan oleh nilai TN dan TP yang tinggi, serta nilai FP dan FN yang relatif rendah.


### C6.3 Menggunakan Classification Report


In [ ]:
# Menampilkan classification report
report = classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)'])
print('Classification Report (tanpa HPO):')
print(report)

**Q9. Jelaskan hasil performansi menggunakan classification report yang anda dapatkan!**

**Jawaban:**

Classification report menampilkan metrik evaluasi per kelas, yaitu:

- **Precision:** Dari semua data yang diprediksi sebagai kelas tertentu, berapa proporsi yang benar. Tinggi = model jarang "salah tuduh".
  $$Precision = \frac{TP}{TP + FP}$$

- **Recall:** Dari semua data yang sebenarnya kelas tertentu, berapa proporsi yang berhasil dideteksi. Tinggi = model jarang "meloloskan" data yang seharusnya terdeteksi.
  $$Recall = \frac{TP}{TP + FN}$$

- **F1-Score:** Rata-rata harmonik dari precision dan recall, memberikan satu metrik yang menyeimbangkan keduanya.
  $$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

- **Support:** Jumlah data aktual per kelas pada data uji.

Dari classification report, dapat dilihat performa model SVM pada masing-masing kelas (No dan Yes). Model ini akan menjadi baseline sebelum diterapkan Hyperparameter Optimization.


# D. Kesimpulan Umum tanpa Menggunakan HPO


**Q10. Setelah melakukan eksplorasi model menggunakan SVM pada dataset Bank Marketing, buatlah hasil pengamatan yang anda temukan dan buat kesimpulan!**

**Jawaban:**

Berdasarkan eksplorasi model SVM tanpa HPO pada dataset Bank Marketing, berikut hasil pengamatan:

1. **Preprocessing sangat penting untuk SVM.** Dataset Bank Marketing memiliki banyak fitur kategorikal yang perlu di-encode menjadi numerik. Selain itu, standarisasi fitur (StandardScaler) sangat berpengaruh terhadap performa SVM karena SVM sensitif terhadap perbedaan skala antar fitur.

2. **Model SVM dengan parameter default** (`kernel='rbf'`, `C=1.0`, `gamma='scale'`) sudah memberikan hasil yang cukup baik pada dataset ini.

3. **Hasil accuracy, precision, recall, dan F1-score** yang diperoleh menjadi baseline yang akan dibandingkan dengan model SVM yang dioptimasi menggunakan HPO (GridSearchCV).

4. Dari confusion matrix, dapat diamati apakah model lebih cenderung melakukan kesalahan tipe False Positive atau False Negative, yang penting untuk konteks bisnis bank dalam menentukan strategi pemasaran.

**Kesimpulan:** Model SVM tanpa HPO sudah mampu mengklasifikasikan nasabah yang akan melakukan deposit atau tidak. Namun, masih ada potensi peningkatan performa melalui optimasi hyperparameter (HPO) pada tahap selanjutnya.


# E. Optimasi: Hyperparameter Tuning dengan GridSearch


## E1. Tahap 1: Pencarian Parameter yang Optimal


In [ ]:
# Import library yang dibutuhkan
from sklearn.model_selection import GridSearchCV

In [ ]:
# Setting parameter untuk GridSearchCV
# Eksplorasi menggunakan 4 parameter SVM
# Referensi: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

param_grid = {
    'C': [0.1, 1, 10, 100],              # Parameter regularisasi
    'gamma': ['scale', 'auto', 0.01, 0.1], # Koefisien kernel
    'kernel': ['rbf', 'linear', 'poly'],   # Jenis fungsi kernel
    'degree': [2, 3, 4]                    # Derajat polinomial (hanya untuk kernel 'poly')
}

print('Parameter Grid yang akan dieksplorasi:')
for key, values in param_grid.items():
    print(f'  {key}: {values}')

# Hitung total kombinasi
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f'\nTotal kombinasi parameter: {total_combinations}')

**Q11. Jelaskan maksud dari code param_grid di atas!**

**Jawaban:**

`param_grid` adalah sebuah dictionary yang berisi kombinasi hyperparameter SVM yang akan dicoba oleh GridSearchCV untuk menemukan kombinasi terbaik. Parameter yang dieksplorasi:

1. **`C` (Regularisasi):** [0.1, 1, 10, 100] — Mengontrol trade-off antara margin yang lebar dan kesalahan klasifikasi. Nilai C yang besar membuat model lebih ketat (batas keputusan lebih kompleks), sedangkan nilai C kecil memberikan margin yang lebih lebar tetapi mungkin lebih banyak misklasifikasi.

2. **`gamma` (Koefisien Kernel):** ['scale', 'auto', 0.01, 0.1] — Menentukan seberapa besar pengaruh satu sampel data. Gamma tinggi = pengaruh lokal (bisa overfitting), gamma rendah = pengaruh global (bisa underfitting).

3. **`kernel` (Fungsi Kernel):** ['rbf', 'linear', 'poly'] — Menentukan cara SVM memetakan data ke ruang dimensi tinggi. RBF untuk data non-linear, linear untuk data yang bisa dipisahkan secara linear, poly untuk pemisahan polinomial.

4. **`degree` (Derajat Polinomial):** [2, 3, 4] — Hanya berlaku untuk kernel `poly`. Menentukan derajat fungsi polinomial yang digunakan.

Total kombinasi yang akan diuji adalah 4 × 4 × 3 × 3 = **144 kombinasi**.


In [ ]:
# Membuat GridSearchCV
grid = GridSearchCV(
    estimator=SVC(random_state=42),  # Model SVM
    param_grid=param_grid,            # Kombinasi parameter
    cv=5,                             # 5-fold cross validation
    scoring='accuracy',               # Metrik evaluasi
    n_jobs=-1,                        # Gunakan semua core CPU
    verbose=1                         # Tampilkan progress
)

print('GridSearchCV berhasil dibuat.')
print(grid)

In [ ]:
# Melakukan training dengan GridSearchCV (catat waktu training)
start_time_hpo = time.time()
grid.fit(X_train_scaled, y_train)
hpo_time = time.time() - start_time_hpo

print(f'\nGridSearch selesai!')
print(f'Waktu training HPO: {hpo_time:.2f} detik ({hpo_time/60:.2f} menit)')

In [ ]:
# Menampilkan parameter terbaik
print('=== Hasil Hyperparameter Optimization ===')
print(f'\nParameter Terbaik:')
for param, value in grid.best_params_.items():
    print(f'  {param}: {value}')
print(f'\nBest Accuracy (CV): {grid.best_score_:.4f} ({grid.best_score_*100:.2f}%)')
print(f'\nWaktu training HPO: {hpo_time:.2f} detik')
print(f'Waktu training tanpa HPO: {training_time:.2f} detik')

**Q12. Jelaskan maksud dan hasil dari code grid, grid.fit(), dan grid.best_params_ di atas!**

**Jawaban:**

1. **`GridSearchCV(...)`:** Membuat objek GridSearchCV yang akan melakukan pencarian parameter terbaik secara exhaustive (mencoba semua kombinasi). Parameter penting:
   - `estimator=SVC(random_state=42)`: Model SVM yang akan dioptimasi.
   - `param_grid=param_grid`: Kombinasi parameter yang akan diuji.
   - `cv=5`: Menggunakan 5-fold cross validation — data latih dibagi menjadi 5 bagian, dan setiap kombinasi parameter diuji 5 kali.
   - `scoring='accuracy'`: Metrik evaluasi yang digunakan untuk memilih parameter terbaik.
   - `n_jobs=-1`: Menggunakan semua core CPU untuk mempercepat proses.

2. **`grid.fit(X_train_scaled, y_train)`:** Menjalankan proses pencarian parameter terbaik. Untuk setiap kombinasi parameter, model dilatih dan dievaluasi menggunakan cross validation. Total proses = 144 kombinasi × 5 fold = 720 kali training.

3. **`grid.best_params_`:** Menampilkan kombinasi parameter yang menghasilkan akurasi tertinggi dari seluruh proses pencarian. Parameter inilah yang akan digunakan untuk membangun model SVM yang dioptimasi.


## E2. Tahap 2: Uji Coba Optimasi Model SVM Menggunakan Parameter HPO


### E2.1 Siapkan Variabel Classifier dan Tentukan Parameternya Berdasarkan Hasil Pencarian Parameter HPO


In [ ]:
# Menentukan variabel classifier dengan parameter terbaik dari HPO
classifier2 = SVC(**grid.best_params_, random_state=42)

print('Classifier SVM (dengan HPO):')
print(classifier2)
print(f'\nParameter yang digunakan: {grid.best_params_}')

**Q13. Jelaskan maksud dari code classifier2 di atas!**

**Jawaban:**

`classifier2` adalah model SVM baru yang dibuat menggunakan parameter terbaik hasil pencarian GridSearchCV. Sintaks `**grid.best_params_` melakukan *unpacking* dictionary parameter terbaik, sehingga setiap parameter otomatis dimasukkan ke dalam konstruktor `SVC()`. Misalnya, jika parameter terbaik adalah `{'C': 10, 'gamma': 'scale', 'kernel': 'rbf', 'degree': 3}`, maka setara dengan `SVC(C=10, gamma='scale', kernel='rbf', degree=3, random_state=42)`.

Model `classifier2` ini berbeda dari `classifier` (model pertama tanpa HPO) karena menggunakan parameter yang sudah dioptimasi, sehingga diharapkan memiliki performa yang lebih baik.


### E2.2 Lakukan Proses Training dengan Data Latih


In [ ]:
# Melakukan training model SVM dengan parameter HPO
start_time_hpo2 = time.time()
classifier2.fit(X_train_scaled, y_train)
training_time_hpo = time.time() - start_time_hpo2

print(f'Training selesai!')
print(f'Waktu training (HPO): {training_time_hpo:.2f} detik')
print(f'Jumlah support vectors: {classifier2.n_support_}')
print(f'Total support vectors: {sum(classifier2.n_support_)}')

**Q14. Jelaskan maksud dari code classifier2.fit() di atas! Mengapa perlu dilakukan training kembali?**

**Jawaban:**

`classifier2.fit(X_train_scaled, y_train)` melakukan proses training ulang menggunakan model SVM dengan parameter terbaik dari HPO.

**Mengapa perlu training kembali?** Karena `classifier2` adalah model SVM **baru** yang belum dilatih. Meskipun GridSearchCV sudah melakukan training selama proses pencarian, model yang dihasilkan oleh GridSearchCV menggunakan cross-validation (membagi data latih menjadi fold-fold). Dengan melakukan training kembali pada **seluruh data latih**, model `classifier2` dapat mempelajari pola dari seluruh data latih (bukan hanya sebagian fold), sehingga diharapkan menghasilkan performa yang lebih optimal.


### E2.3 Lakukan Pengujian dengan Data Uji


**Q15. Jelaskan maksud dan hasil dari code pengujian di bawah ini!**

**Jawaban:**

Code pengujian di bawah melakukan prediksi pada data uji menggunakan model SVM yang sudah dioptimasi (classifier2). Hasil prediksi dibandingkan dengan label asli dalam bentuk tabel perbandingan untuk melihat seberapa akurat model HPO dalam mengklasifikasikan data. Hasilnya diharapkan lebih baik dibandingkan model tanpa HPO karena parameter yang digunakan sudah dioptimasi.


In [ ]:
# Melakukan prediksi pada data uji dengan model HPO
y_pred2 = classifier2.predict(X_test_scaled)

# Perbandingan data prediksi dengan label asli
comparison2 = pd.DataFrame({
    'Data Ke': range(1, len(y_test) + 1),
    'Label Asli': y_test.values,
    'Hasil Prediksi': y_pred2,
    'Keterangan': ['Benar' if a == p else 'Salah' for a, p in zip(y_test.values, y_pred2)]
})

print('Perbandingan Label Asli vs Hasil Prediksi HPO (20 data pertama):')
print(comparison2.head(20).to_string(index=False))

# Menghitung jumlah prediksi benar dan salah
benar2 = (comparison2['Keterangan'] == 'Benar').sum()
salah2 = (comparison2['Keterangan'] == 'Salah').sum()
print(f'\nJumlah prediksi benar: {benar2} dari {len(y_test)} data')
print(f'Jumlah prediksi salah: {salah2} dari {len(y_test)} data')

## E3. Tahap 3: Analisa Performansi Model


### E3.1 Menggunakan Accuracy Score


In [ ]:
# Menghitung skor akurasi model HPO
accuracy2 = accuracy_score(y_test, y_pred2)
print(f'Accuracy Score (dengan HPO): {accuracy2:.4f}')
print(f'Accuracy Percentage: {accuracy2 * 100:.2f}%')
print(f'\nPerbandingan:')
print(f'  Tanpa HPO: {accuracy * 100:.2f}%')
print(f'  Dengan HPO: {accuracy2 * 100:.2f}%')
print(f'  Selisih: {(accuracy2 - accuracy) * 100:+.2f}%')

**Q16. Jelaskan hasil performansi menggunakan accuracy score yang anda dapatkan!**

**Jawaban:**

Accuracy score model SVM dengan HPO menunjukkan persentase prediksi yang benar dari total data uji. Dibandingkan dengan model tanpa HPO, accuracy model dengan HPO menunjukkan ada/tidaknya peningkatan performa setelah dilakukan optimasi hyperparameter. Selisih accuracy antara kedua model menunjukkan kontribusi HPO terhadap peningkatan performa model.


### E3.2 Menggunakan Confusion Matrix


In [ ]:
# Menampilkan confusion matrix model HPO
cm2 = confusion_matrix(y_test, y_pred2)
print('Confusion Matrix (dengan HPO):')
print(cm2)

# Visualisasi confusion matrix perbandingan
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix tanpa HPO
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No (0)', 'Yes (1)'])
disp1.plot(cmap='Reds', ax=axes[0], values_format='d')
axes[0].set_title('Confusion Matrix\nTanpa HPO', fontsize=13, fontweight='bold')

# Confusion Matrix dengan HPO
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=['No (0)', 'Yes (1)'])
disp2.plot(cmap='Blues', ax=axes[1], values_format='d')
axes[1].set_title('Confusion Matrix\nDengan HPO', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/confusion_matrix_perbandingan.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretasi
tn2, fp2, fn2, tp2 = cm2.ravel()
print(f'\nTrue Negative (TN): {tn2}')
print(f'False Positive (FP): {fp2}')
print(f'False Negative (FN): {fn2}')
print(f'True Positive (TP): {tp2}')

**Q17. Jelaskan hasil performansi menggunakan confusion matrix yang anda dapatkan!**

**Jawaban:**

Confusion matrix model SVM dengan HPO dibandingkan secara visual (side by side) dengan confusion matrix tanpa HPO. Dari perbandingan tersebut, dapat dilihat perubahan jumlah True Negative, False Positive, False Negative, dan True Positive.

Peningkatan performa ditunjukkan oleh:
- Meningkatnya nilai TN dan TP (prediksi benar)
- Menurunnya nilai FP dan FN (prediksi salah)

Jika terjadi peningkatan pada kedua indikator tersebut, berarti optimasi hyperparameter berhasil meningkatkan kemampuan model dalam mengklasifikasikan kedua kelas (No dan Yes) dengan lebih akurat.


### E3.3 Menggunakan Classification Report


In [ ]:
# Menampilkan classification report model HPO
report2 = classification_report(y_test, y_pred2, target_names=['No (0)', 'Yes (1)'])
print('Classification Report (dengan HPO):')
print(report2)

**Q18. Jelaskan hasil performansi menggunakan classification report yang anda dapatkan!**

**Jawaban:**

Classification report model SVM dengan HPO menampilkan precision, recall, dan F1-score per kelas, serta weighted average. Dengan membandingkan classification report sebelum dan sesudah HPO, dapat dilihat apakah optimasi hyperparameter berhasil meningkatkan:

- **Precision:** Kemampuan model untuk tidak salah memprediksi kelas positif/negatif.
- **Recall:** Kemampuan model untuk mendeteksi seluruh data pada masing-masing kelas.
- **F1-Score:** Keseimbangan antara precision dan recall.

Peningkatan pada metrik-metrik ini menunjukkan bahwa HPO berhasil menemukan kombinasi parameter yang lebih optimal untuk model SVM pada dataset Bank Marketing.


# F. Kesimpulan Umum Menggunakan HPO


In [ ]:
# Membuat tabel perbandingan performansi
report_dict1 = classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)'], output_dict=True)
report_dict2 = classification_report(y_test, y_pred2, target_names=['No (0)', 'Yes (1)'], output_dict=True)

comparison_table = pd.DataFrame({
    'Metrik': ['Accuracy', 'Precision (weighted avg)', 'Recall (weighted avg)', 'F1-Score (weighted avg)', 'Waktu Training (detik)'],
    'Tanpa HPO': [
        f'{accuracy:.4f} ({accuracy*100:.2f}%)',
        f"{report_dict1['weighted avg']['precision']:.4f}",
        f"{report_dict1['weighted avg']['recall']:.4f}",
        f"{report_dict1['weighted avg']['f1-score']:.4f}",
        f'{training_time:.2f}'
    ],
    'Dengan HPO': [
        f'{accuracy2:.4f} ({accuracy2*100:.2f}%)',
        f"{report_dict2['weighted avg']['precision']:.4f}",
        f"{report_dict2['weighted avg']['recall']:.4f}",
        f"{report_dict2['weighted avg']['f1-score']:.4f}",
        f'{training_time_hpo:.2f}'
    ]
})

print('=== TABEL PERBANDINGAN PERFORMANSI ===')
print('='*70)
print(comparison_table.to_string(index=False))
print('='*70)

print(f'\nParameter SVM Tanpa HPO: kernel=rbf, C=1.0, gamma=scale')
print(f'Parameter SVM Dengan HPO: {grid.best_params_}')
print(f'Waktu pencarian HPO (GridSearchCV): {hpo_time:.2f} detik ({hpo_time/60:.2f} menit)')

In [ ]:
# Visualisasi perbandingan performansi
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
tanpa_hpo = [
    accuracy,
    report_dict1['weighted avg']['precision'],
    report_dict1['weighted avg']['recall'],
    report_dict1['weighted avg']['f1-score']
]
dengan_hpo = [
    accuracy2,
    report_dict2['weighted avg']['precision'],
    report_dict2['weighted avg']['recall'],
    report_dict2['weighted avg']['f1-score']
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, tanpa_hpo, width, label='Tanpa HPO', color='#e74c3c', edgecolor='black')
bars2 = ax.bar(x + width/2, dengan_hpo, width, label='Dengan HPO', color='#2ecc71', edgecolor='black')

ax.set_ylabel('Skor', fontsize=12)
ax.set_title('Perbandingan Performansi SVM: Tanpa HPO vs Dengan HPO', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)

# Tambahkan nilai di atas bar
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.4f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('plots/perbandingan_performansi.png', dpi=150, bbox_inches='tight')
plt.show()

**Q19. Buatlah hasil perbandingan performansi, sebelum dan sesudah penggunaan HPO. Buatlah kesimpulan yang anda dapatkan dari hasil perbandingan tersebut.**

**Jawaban:**

Berikut hasil perbandingan performansi model SVM sebelum dan sesudah penggunaan Hyperparameter Optimization (HPO) menggunakan GridSearchCV:

**Perbandingan:**
- Tabel perbandingan performansi dan grafik bar chart sudah ditampilkan di atas.
- Parameter terbaik hasil HPO telah ditampilkan.

**Kesimpulan:**

1. **Hyperparameter Optimization (HPO) menggunakan GridSearchCV** berhasil menemukan kombinasi parameter terbaik untuk model SVM pada dataset Bank Marketing.

2. **Perbandingan Accuracy:** Accuracy model SVM dengan HPO dibandingkan dengan model tanpa HPO menunjukkan ada/tidaknya peningkatan. Peningkatan ini menunjukkan bahwa pemilihan parameter yang tepat berpengaruh signifikan terhadap performa model.

3. **Trade-off waktu vs performa:** Proses GridSearchCV membutuhkan waktu yang jauh lebih lama dibandingkan training model tunggal karena harus mencoba banyak kombinasi parameter dengan cross-validation. Namun, investasi waktu ini sepadan jika terjadi peningkatan performa yang signifikan.

4. **Preprocessing penting:** Standarisasi data (StandardScaler) dan encoding fitur kategorikal sangat berpengaruh terhadap performa SVM. Tanpa preprocessing yang tepat, model SVM tidak akan optimal.

5. **Model SVM cocok untuk dataset Bank Marketing** karena mampu menangani data dengan banyak fitur (16 fitur) dan dataset yang cukup besar (11.162 data) dengan baik.
